# 23.4 Mechanism design & auctions

Mechanism design writes the rules so that self-interested choices reveal the information the system needs. Auctions turn Nash reasoning inside out: expected utility drives bidding, allocation rules pick winners, and payment rules shape incentives. Save a copy to Drive to edit.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 234
random.seed(SEED)
np.random.seed(SEED)

## The concept, built once: second-price allocation and payment

In a single-item second-price auction, the highest bidder wins and pays the highest losing bid, subject to a reserve. With lesson bids $(10,7,4)$, bidder A wins, pays $7$, and has utility $10-7=3$.

In [ ]:
def second_price_auction(bids, values=None, reserve=0.0, tie_rule="lexicographic"):
    names = list(bids.keys())
    eligible = [name for name in names if bids[name] >= reserve]
    if not eligible:
        return {"winner": None, "price": 0.0, "utility": 0.0, "revenue": 0.0}

    if tie_rule == "lexicographic":
        eligible.sort(key=lambda name: (-bids[name], name))
    else:
        eligible.sort(key=lambda name: -bids[name])

    winner = eligible[0]
    losing_bids = [bids[name] for name in names if name != winner and bids[name] >= reserve]
    second_price = max(losing_bids, default=reserve)
    price = max(reserve, second_price)
    value = bids[winner] if values is None else values[winner]
    utility = value - price
    return {"winner": winner, "price": float(price), "utility": float(utility), "revenue": float(price)}


lesson_bids = {"A": 10, "B": 7, "C": 4}
lesson_values = {"A": 10, "B": 7, "C": 4}
lesson_outcome = second_price_auction(lesson_bids, lesson_values)
print(lesson_outcome)
assert lesson_outcome["winner"] == "A"
assert lesson_outcome["price"] == 7.0
assert lesson_outcome["utility"] == 3.0

Reserve prices change the payment rule. For the same bids, $r=0$ gives revenue $7$, $r=8$ gives revenue $8$, and $r=11$ gives revenue $0$ because no bid clears the reserve.

First price is different: utility is $v_i-b_i$ when the bidder wins, so a value-10 bidder earns $2$ by bidding $8$ against opponents $7$ and $4$, but earns $0$ by bidding truthfully at $10$.

In [ ]:
def first_price_utility(value, bid, opponent_bids, reserve=0.0):
    all_opponents = list(opponent_bids)
    if bid < reserve:
        return 0.0
    if all_opponents and bid <= max(all_opponents):
        return 0.0
    return float(value - bid)


revenues = {}
for reserve in [0, 8, 11]:
    revenues[reserve] = second_price_auction(lesson_bids, lesson_values, reserve=reserve)["revenue"]

shaded_utility = first_price_utility(10, 8, [7, 4])
truthful_first_price_utility = first_price_utility(10, 10, [7, 4])
print(revenues, shaded_utility, truthful_first_price_utility)
assert revenues[0] == 7.0
assert revenues[8] == 8.0
assert revenues[11] == 0.0
assert shaded_utility == 2.0
assert truthful_first_price_utility == 0.0

## The dataset ladder: D1 to D5 auction profiles

The ladder grows from the 3-bidder lesson auction to more bidders, reserve conflicts, multi-unit ties, and a hardest first-price manipulation/shading profile.

In [ ]:
def uniform_price_multiunit(bids, units, reserve=0.0):
    eligible = [(name, bid) for name, bid in bids.items() if bid >= reserve]
    eligible.sort(key=lambda item: (-item[1], item[0]))
    winners = eligible[:units]
    losers = eligible[units:]
    clearing = losers[0][1] if losers else reserve
    price = max(reserve, clearing)
    return {"winners": [name for name, bid in winners], "price": float(price), "revenue": float(price * len(winners))}


def build_auction_ladder():
    return [
        {"name": "D1 lesson second-price", "bids": {"A": 10, "B": 7, "C": 4}, "values": {"A": 10, "B": 7, "C": 4}, "reserve": 0, "kind": "second"},
        {"name": "D2 five-bidder second-price", "bids": {"A": 12, "B": 11, "C": 8, "D": 5, "E": 2}, "values": {"A": 12, "B": 11, "C": 8, "D": 5, "E": 2}, "reserve": 0, "kind": "second"},
        {"name": "D3 reserve pressure", "bids": {"A": 10, "B": 7, "C": 4}, "values": {"A": 10, "B": 7, "C": 4}, "reserve": 8, "kind": "second"},
        {"name": "D4 multi-unit ties", "bids": {"A": 9, "B": 9, "C": 6, "D": 6, "E": 3, "F": 1}, "values": {"A": 10, "B": 9, "C": 7, "D": 6, "E": 3, "F": 1}, "reserve": 2, "kind": "uniform", "units": 2},
        {"name": "D5 first-price manipulation", "bids": {"A": 8, "B": 7, "C": 4}, "values": {"A": 10, "B": 7, "C": 4}, "reserve": 0, "kind": "first"},
    ]


auction_ladder = build_auction_ladder()
for rung in auction_ladder:
    print(rung["name"], "bidders", len(rung["bids"]), "reserve", rung["reserve"], "kind", rung["kind"])

## Run the SAME method across D1-D5

Apply the appropriate auction mechanism on every rung and collect winner, revenue, and surplus.

In [ ]:
def first_price_outcome(bids, values, reserve=0.0):
    eligible = [(name, bid) for name, bid in bids.items() if bid >= reserve]
    if not eligible:
        return {"winner": None, "price": 0.0, "utility": 0.0, "revenue": 0.0}
    eligible.sort(key=lambda item: (-item[1], item[0]))
    winner, price = eligible[0]
    utility = values[winner] - price
    return {"winner": winner, "price": float(price), "utility": float(utility), "revenue": float(price)}


def truthful_surplus(outcome, values):
    if outcome["winner"] is None:
        return 0.0
    return float(values[outcome["winner"]] - outcome["price"])


auction_results = []
for rung in auction_ladder:
    if rung["kind"] == "uniform":
        outcome = uniform_price_multiunit(rung["bids"], rung["units"], reserve=rung["reserve"])
        revenue = outcome["revenue"]
        winner = ",".join(outcome["winners"])
        surplus = sum(rung["values"][name] - outcome["price"] for name in outcome["winners"])
    elif rung["kind"] == "first":
        outcome = first_price_outcome(rung["bids"], rung["values"], reserve=rung["reserve"])
        revenue = outcome["revenue"]
        winner = outcome["winner"]
        surplus = truthful_surplus(outcome, rung["values"])
    else:
        outcome = second_price_auction(rung["bids"], rung["values"], reserve=rung["reserve"])
        revenue = outcome["revenue"]
        winner = outcome["winner"]
        surplus = truthful_surplus(outcome, rung["values"])
    auction_results.append({
        "name": rung["name"],
        "bidders": len(rung["bids"]),
        "winner": winner,
        "revenue": revenue,
        "surplus": surplus,
        "outcome": outcome,
    })

for row in auction_results:
    print(row["name"], "winner", row["winner"], "revenue", row["revenue"], "surplus", row["surplus"])

## Results visualization

The closing figure has bid/payment panels plus revenue and surplus curves over bidder count.

In [ ]:
fig, axes = plt.subplots(2, len(auction_ladder), figsize=(18, 7))

for axis, rung, row in zip(axes[0], auction_ladder, auction_results):
    names = list(rung["bids"].keys())
    bids = [rung["bids"][name] for name in names]
    axis.bar(names, bids)
    axis.axhline(row["revenue"], color="tab:red", linestyle="--", label="payment or revenue")
    axis.set_title(rung["name"].split()[0] + " bids")
    axis.legend(fontsize=8)

bidders = [row["bidders"] for row in auction_results]
revenues = [row["revenue"] for row in auction_results]
surpluses = [row["surplus"] for row in auction_results]
axes[1, 0].plot(bidders, revenues, marker="o", label="revenue")
axes[1, 0].set_xlabel("bidders")
axes[1, 0].set_ylabel("revenue")
axes[1, 0].legend()
axes[1, 1].plot(bidders, surpluses, marker="s", color="tab:purple", label="winner surplus")
axes[1, 1].set_xlabel("bidders")
axes[1, 1].set_ylabel("surplus")
axes[1, 1].legend()
for axis in axes[1, 2:]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Pitfall on D5: assuming truthfulness in first-price auctions

The lesson's pitfall is direct: in a first-price auction with value 10 and opponents 7 and 4, truthful bid 10 gives utility 0 while shaded bid 8 gives utility 2. The fix is to use a second-price rule when truthfulness is desired, or model shading explicitly.

In [ ]:
truthful_utility = first_price_utility(10, 10, [7, 4])
shaded_utility = first_price_utility(10, 8, [7, 4])
second_price_truthful = second_price_auction({"A": 10, "B": 7, "C": 4}, {"A": 10, "B": 7, "C": 4})
print("first-price truthful utility", truthful_utility)
print("first-price shaded utility", shaded_utility)
print("second-price truthful utility", second_price_truthful["utility"])
assert truthful_utility == 0.0
assert shaded_utility == 2.0
assert second_price_truthful["utility"] == 3.0

## Evaluate it + Practice

- Metric: revenue or truthful surplus; compare with a no-reserve or random-winner baseline.
- Sanity check: in second price, the winner's payment cannot exceed the winner's bid unless a reserve above bid blocks allocation.
- Ablation: remove deterministic tie-breaking in D4 and the allocation is not reproducible.
- Failure signal: first-price truthful bidding has avoidable regret when a lower winning bid is available.

Practice 1: Sweep reserves from 0 to 12 for the D1 bids and plot revenue.

Practice 2: Change D4 to three units and recompute winners, clearing price, and revenue.